# Banco de Dados de Análise de Crédito

## Objetivo
 
Construir um modelo de classificação binária para prever o **risco de crédito** de clientes,
com base em variáveis financeiras, pessoais e comportamentais.
 
| | |
|---|---|
| **Variável Alvo** | `risco_credito` |
| **Tipo** | Binária: `0` = bom pagador, `1` = mau pagador |
| **Tarefa** | Classificação supervisionada |
 
---
 
## Fonte dos Dados
 
- Dataset: German Credit Data
- Arquivo: `german_credit_data.csv`
- Armazenamento: banco SQLite (`analise_credito_alemao.db`), tabela `credito_clientes`
---
 
## Variáveis do Dataset
 
### Numéricas
| Coluna | Descrição |
|---|---|
| `duracao_meses` | Duração do crédito em meses |
| `valor_credito` | Valor do crédito solicitado |
| `idade` | Idade do cliente |
 
### Categóricas Ordinais
| Coluna | Descrição |
|---|---|
| `status_conta_corrente` | Situação da conta corrente |
| `poupanca_investimento` | Nível de poupança/investimento |
| `tempo_emprego_atual` | Tempo no emprego atual |
| `taxa_parcelamento_renda` | Taxa de parcelamento em relação à renda |
| `residencia_atual_desde` | Tempo na residência atual |
| `numero_creditos_existentes` | Quantidade de créditos ativos |
| `dependentes` | Número de pessoas dependentes |
 
### Categóricas Nominais
| Coluna | Descrição |
|---|---|
| `historico_credito` | Histórico de pagamentos |
| `proposito` | Finalidade do crédito |
| `status_pessoal_sexo` | Estado civil e sexo |
| `outros_fiadores` | Existência de fiadores |
| `propriedade` | Tipo de propriedade |
| `outros_planos_parcelamento` | Outros planos de parcelamento |
| `habitacao` | Tipo de habitação |
| `trabalho` | Cargo/ocupação |
| `telefone` | Possui telefone |
| `trabalhador_estrangeiro` | É trabalhador estrangeiro |
 
---

In [9]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from importlib import reload
import sqlite3 # Usar SQLite para facilitar o upload de um arquivo único ao GitHub

In [18]:
# Carregar dados
df = pd.read_csv('data/german_credit_data.csv')

# Dicionário de tradução (ajuste conforme sua interpretação das variáveis)
colunas_pt = {
    'laufkont': 'status_conta_corrente',
    'laufzeit': 'duracao_meses',
    'moral': 'historico_credito',
    'verw': 'proposito',
    'hoehe': 'valor_credito',
    'sparkont': 'poupanca_investimento',
    'beszeit': 'tempo_emprego_atual',
    'rate': 'taxa_parcelamento_renda',
    'famges': 'status_pessoal_sexo',
    'buerge': 'outros_fiadores',
    'wohnzeit': 'residencia_atual_desde',
    'verm': 'propriedade',
    'alter': 'idade',
    'weitkred': 'outros_planos_parcelamento',
    'wohn': 'habitacao',
    'bishkred': 'numero_creditos_existentes',
    'beruf': 'trabalho',
    'pers': 'dependentes',
    'telef': 'telefone',
    'gastarb': 'trabalhador_estrangeiro',
    'kredit': 'risco_credito'
}

df.rename(columns=colunas_pt, inplace=True)

# Criando o banco de dados local (gerando um arquivo .db)
conn = sqlite3.connect('analise_credito_alemao.db')
df.to_sql('credito_cliente', conn, if_exists='replace', index=False)

1081

In [15]:
def run_query(query):
    return pd.read_sql_query(query, conn)

In [ ]:
query = """
SELECT
    -- ========================
    -- VARIÁVEIS NUMÉRICAS
    -- ========================

    CASE 
        WHEN duracao_meses <= 0 THEN NULL
        ELSE duracao_meses 
    END AS duracao_meses,

    CASE 
        WHEN valor_credito <= 0 THEN NULL
        ELSE valor_credito 
    END AS valor_credito,

    CASE 
        WHEN idade < 18 OR idade > 100 THEN NULL
        ELSE idade 
    END AS idade,

    -- ========================
    -- VARIÁVEL ALVO
    -- ========================

    CASE 
        WHEN risco_credito NOT IN (0,1) THEN NULL
        ELSE risco_credito 
    END AS risco_credito,

    -- ========================
    -- CATEGÓRICAS (com cast para INTEGER)
    -- ========================
    CAST(status_conta_corrente AS INTEGER) AS status_conta_corrente,
    CAST(historico_credito AS INTEGER) AS historico_credito,
    CAST(proposito AS INTEGER) AS proposito,
    CAST(poupanca_investimento AS INTEGER) AS poupanca_investimento,
    CAST(tempo_emprego_atual AS INTEGER) AS tempo_emprego_atual,
    CAST(taxa_parcelamento_renda AS INTEGER) AS taxa_parcelamento_renda,
    CAST(status_pessoal_sexo AS INTEGER) AS status_pessoal_sexo,
    CAST(outros_fiadores AS INTEGER) AS outros_fiadores,
    CAST(residencia_atual_desde AS INTEGER) AS residencia_atual_desde,
    CAST(propriedade AS INTEGER) AS propriedade,
    CAST(outros_planos_parcelamento AS INTEGER) AS outros_planos_parcelamento,
    CAST(habitacao AS INTEGER) AS habitacao,
    CAST(numero_creditos_existentes AS INTEGER) AS numero_creditos_existentes,
    CAST(trabalho AS INTEGER) AS trabalho,
    CAST(dependentes AS INTEGER) AS dependentes,
    CAST(telefone AS INTEGER) AS telefone,
    CAST(trabalhador_estrangeiro AS INTEGER) AS trabalhador_estrangeiro

FROM credito_cliente

-- Foco apenas no target
WHERE risco_credito IN (0,1)
"""

df_limpo = run_query(query)

print(f"\nNulos por coluna:\n{df_limpo.isnull().sum()}")
print(f"\nValores Duplicados:\n{df_limpo.duplicated().sum()}")

df_limpo = df_limpo.dropna()
df_limpo = df_limpo.drop_duplicates()

print(f"Linhas originais  : {len(run_query('SELECT * FROM credito_cliente'))}")
print(f"Linhas após limpeza: {len(df_limpo)}")



Nulos por coluna:
duracao_meses                 0
valor_credito                 0
idade                         0
risco_credito                 0
status_conta_corrente         0
historico_credito             0
proposito                     0
poupanca_investimento         0
tempo_emprego_atual           0
taxa_parcelamento_renda       0
status_pessoal_sexo           0
outros_fiadores               0
residencia_atual_desde        0
propriedade                   0
outros_planos_parcelamento    0
habitacao                     0
numero_creditos_existentes    0
trabalho                      0
dependentes                   0
telefone                      0
trabalhador_estrangeiro       0
dtype: int64

Valores Duplicados:
0
Linhas originais  : 1000
Linhas após limpeza: 1000
Linhas originais  : 1000
Linhas após limpeza: 1000

Nulos por coluna:
duracao_meses                 0
valor_credito                 0
idade                         0
risco_credito                 0
status_conta_corrente   

In [29]:
df_limpo.head()

,duracao_meses,valor_credito,idade,risco_credito,status_conta_corrente,historico_credito,proposito,poupanca_investimento,tempo_emprego_atual,taxa_parcelamento_renda,status_pessoal_sexo,outros_fiadores,residencia_atual_desde,propriedade,outros_planos_parcelamento,habitacao,numero_creditos_existentes,trabalho,dependentes,telefone,trabalhador_estrangeiro
0,18,1049,21,1,1,4,2,1,2,4,2,1,4,2,3,1,1,3,2,1,2
1,9,2799,36,1,1,4,0,1,3,2,3,1,2,1,3,1,2,3,1,1,2
2,12,841,23,1,2,2,9,2,4,2,2,1,4,1,3,1,1,2,2,1,2
3,12,2122,39,1,1,4,0,1,3,3,3,1,2,1,3,1,2,2,1,1,1
4,12,2171,38,1,1,4,0,1,3,4,3,1,4,2,1,2,2,2,2,1,1


# Resumo dos Ajustes - Análise de Crédito Alemão

## 1. Carregamento dos Dados

- Dados carregados via `pd.read_csv` a partir da pasta `data/`
- Caminho correto: `data/german_credit_data.csv` (relativo ao notebook)
- Colunas renomeadas do alemão para português via dicionário `colunas_pt`

---

## 2. Banco de Dados

- Banco criado com SQLite via `sqlite3.connect('analise_credito_alemao.db')`
- Tabela criada automaticamente com `df.to_sql` — arquivo SQL separado descartado por ser redundante
- Nome padronizado da tabela: `credito_clientes` 

---

## 3. Limpeza dos Dados via SQL

### Variáveis Numéricas
| Coluna | Tratativa | Motivo |
|---|---|---|
| `duracao_meses` | Negativo ou zero vira `NULL` | Duração não pode ser inválida |
| `valor_credito` | Negativo ou zero vira `NULL` | Valor não pode ser inválido |
| `idade` | Fora de 18–100 vira `NULL` | Idades impossíveis |

### Variável Alvo
| Coluna | Tratativa | Motivo |
|---|---|---|
| `risco_credito` | Apenas `0` ou `1` aceitos | Variável binária, não pode ter outro valor |

### Categóricas
- Sem validação por ora
- Valores únicos precisam ser verificados via Python antes de aplicar regras

### Cláusula WHERE
- Foco apenas no target: `WHERE risco_credito IN (0,1)`
- Demais tratativas ficam no `CASE` — transformam valores inválidos em `NULL` sem remover a linha

---